In [1]:
import pandas as pd
import fastf1
from pathlib import Path

fastf1.Cache.enable_cache("../cache")
ROOT_DIR = Path.cwd().parent.resolve()
ROOT_DIR

WindowsPath('D:/Projects/formula1-data-analysis')

In [ ]:
year = 2025
raw_folder = (ROOT_DIR/"data"/"raw"/str(year))

races = [race for race in raw_folder.iterdir() if race.is_dir()]
all_degradation_metrics = []
for race in races:
    race_name = race.name
    
    laps_file = race/"laps.csv"
    
    laps = pd.read_csv(laps_file)
    laps = laps[laps["LapTime"].notna()]
    laps = pd.to_timedelta(laps["LapTime"]).dt.total_seconds
    stint = laps.group_by(
        ["Driver","Stint","Compound"]
    )
    
    for (driver,stint,compound), stint_lap in stint:
        if len(stint_lap) < 6:
            continue
        stint_laps = stint_lap.sort_values("LapNumber")
        avg_pace = stint_laps["LapTime"].mean()
        first_three_avg = stint_laps.head(3)["LapTime"].mean()
        last_three_avg = stint_laps.tail(3)["LapTime"].mean()
        pace_dropoff = last_three_avg-first_three_avg
        all_degradation_metrics.append(
            {
                "Race" : race_name,
                "Driver" : driver,
                "Compound" : compound,
                "Stint" : stint,
                "StintLength" : len(stint_laps),
                "AvgPace" : avg_pace,
                "FirstThreeAvg" : first_three_avg,
                "LastThreeAvg" : last_three_avg,
                "PaceDropoff" : pace_dropoff
            }
        )        
    
    tire_degradation = pd.DataFrame(all_degradation_metrics)
    tire_degradation.head()
    
    

AttributeError: 'function' object has no attribute 'groupby'